In [1]:
!pip install opencv-python mediapipe pillow numpy

In [8]:
!python -m pip install --upgrade pip



  Using cached pip-25.3-py3-none-any.whl.metadata (4.7 kB)
Using cached pip-25.3-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2


In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import time
from collections import deque # For storing rep history

# Initialize MediaPipe drawing and pose solutions
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose

# --- Helper Function to Calculate Angle ---
def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    radians = np.arctan2(c[1] - b[1], c[0] - b[0]) - np.arctan2(a[1] - b[1], a[0] - b[0])
    angle = np.abs(radians * 180.0 / np.pi)

    if angle > 180.0:
        angle = 360 - angle

    return angle

# --- UI Drawing Functions ---
def draw_text_with_background(img, text, x, y, font, font_scale, text_color, bg_color, thickness):
    text_size = cv2.getTextSize(text, font, font_scale, thickness)[0]
    # Adjust background rectangle for better padding
    bg_x1 = x - 5
    bg_y1 = y - text_size[1] - 5
    bg_x2 = x + text_size[0] + 5
    bg_y2 = y + 5
    cv2.rectangle(img, (bg_x1, bg_y1), (bg_x2, bg_y2), bg_color, -1)
    cv2.putText(img, text, (x, y), font, font_scale, text_color, thickness, cv2.LINE_AA)

# --- Exercise Configuration ---
EXERCISE_PARAMS = {
    "bicep_curls": {
        "joint_landmarks": {
            "p1": mp_pose.PoseLandmark.LEFT_SHOULDER,
            "p2": mp_pose.PoseLandmark.LEFT_ELBOW,
            "p3": mp_pose.PoseLandmark.LEFT_WRIST
        },
        "vertex_landmark": mp_pose.PoseLandmark.LEFT_ELBOW,
        "down_angle_threshold": 160, # Angle when arm is fully extended (high angle)
        "up_angle_threshold": 30,    # Angle when arm is fully flexed (low angle)
        "perfect_min_flex_angle": 25,
        "perfect_max_flex_angle": 35,
        "perfect_min_ext_angle": 160,
        "perfect_max_ext_angle": 175,
        "movement_direction": "flex_to_extend" # Curls: starts extended, flexes, then extends to count rep
    },
    "squats": {
        "joint_landmarks": {
            "p1": mp_pose.PoseLandmark.LEFT_HIP,
            "p2": mp_pose.PoseLandmark.LEFT_KNEE,
            "p3": mp_pose.PoseLandmark.LEFT_ANKLE
        },
        "vertex_landmark": mp_pose.PoseLandmark.LEFT_KNEE,
        "down_angle_threshold": 90,  # Angle when fully squatted (low angle)
        "up_angle_threshold": 160,   # Angle when standing (high angle)
        "perfect_min_flex_angle": 75,
        "perfect_max_flex_angle": 95,
        "perfect_min_ext_angle": 165,
        "perfect_max_ext_angle": 175,
        "movement_direction": "extend_to_flex" # Squats: starts extended, flexes, then extends to count rep
    },
    "pushups": {
        "joint_landmarks": {
            "p1": mp_pose.PoseLandmark.LEFT_SHOULDER,
            "p2": mp_pose.PoseLandmark.LEFT_ELBOW,
            "p3": mp_pose.PoseLandmark.LEFT_WRIST
        },
        "vertex_landmark": mp_pose.PoseLandmark.LEFT_ELBOW,
        "down_angle_threshold": 80,  # Angle when arms are bent (low angle)
        "up_angle_threshold": 160,   # Angle when arms are extended (high angle)
        "perfect_min_flex_angle": 70,
        "perfect_max_flex_angle": 90,
        "perfect_min_ext_angle": 160,
        "perfect_max_ext_angle": 175,
        "movement_direction": "extend_to_flex" # Pushups: starts extended, flexes, then extends to count rep
    }
}

# --- Main Program ---
def main():
    cap = cv2.VideoCapture(0)

    if not cap.isOpened():
        print("Error: Could not open video stream. Check camera connection or index.")
        return

    # --- Global State Variables ---
    current_exercise_key = "bicep_curls" # <--- CHOOSE YOUR EXERCISE HERE
    # current_exercise_key = "squats"
    # current_exercise_key = "pushups"

    target_reps = 10
    reps_completed = 0
    stage = None # "up" or "down"
    current_angle = 0
    feedback_message = ""
    feedback_display_start_time = 0
    feedback_display_duration = 2 # seconds

    # Variables to store angles at peak flexion/extension for feedback
    angle_at_flexion = 0
    angle_at_extension = 0

    # Store history for post-set review
    rep_history = deque() # Stores dicts like {"rep_num": 1, "flex_angle": 30, "ext_angle": 170, "feedback": "PERFECT!"}

    # Get specific parameters for the chosen exercise
    params = EXERCISE_PARAMS[current_exercise_key]

    # UI Colors
    WHITE = (255, 255, 255)
    BLACK = (0, 0, 0)
    GREEN = (0, 255, 0)
    RED = (0, 0, 255)
    YELLOW = (0, 255, 255)
    BLUE = (255, 0, 0)
    ORANGE = (0, 165, 255)
    LIGHT_BLUE = (230, 200, 0) # For status box

    ## Setup MediaPipe Pose instance
    with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                print("Ignoring empty camera frame.")
                break

            frame = cv2.flip(frame, 1) # Flip image horizontally
            h, w, c = frame.shape

            image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image.flags.writeable = False
            results = pose.process(image)
            image.flags.writeable = True
            image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

            # --- Exercise Logic and Rep Counting ---
            if reps_completed < target_reps:
                try:
                    landmarks = results.pose_landmarks.landmark

                    p1 = [landmarks[params["joint_landmarks"]["p1"].value].x, landmarks[params["joint_landmarks"]["p1"].value].y]
                    p2 = [landmarks[params["joint_landmarks"]["p2"].value].x, landmarks[params["joint_landmarks"]["p2"].value].y]
                    p3 = [landmarks[params["joint_landmarks"]["p3"].value].x, landmarks[params["joint_landmarks"]["p3"].value].y]

                    current_angle = calculate_angle(p1, p2, p3)

                    vertex_coords = np.multiply([landmarks[params["vertex_landmark"].value].x,
                                                  landmarks[params["vertex_landmark"].value].y],
                                                 [w, h]).astype(int)

                    # --- Repetition Counting Logic with Feedback ---
                    if params["movement_direction"] == "flex_to_extend": # e.g., Bicep Curls
                        if current_angle > params["down_angle_threshold"]: # Extended arm
                            if stage == "up": # Transition from flexed to extended
                                reps_completed += 1
                                # Determine feedback
                                flex_ok = params["perfect_min_flex_angle"] <= angle_at_flexion <= params["perfect_max_flex_angle"]
                                ext_ok = params["perfect_min_ext_angle"] <= angle_at_extension <= params["perfect_max_ext_angle"]
                                
                                rep_feedback = "GOOD!"
                                if flex_ok and ext_ok: rep_feedback = "PERFECT!"
                                elif not flex_ok and angle_at_flexion < params["perfect_min_flex_angle"]: rep_feedback = "NOT DEEP ENOUGH (FLEX)"
                                elif not flex_ok and angle_at_flexion > params["perfect_max_flex_angle"]: rep_feedback = "TOO DEEP (FLEX)"
                                elif not ext_ok and angle_at_extension < params["perfect_min_ext_angle"]: rep_feedback = "NOT EXTENDED ENOUGH"
                                elif not ext_ok and angle_at_extension > params["perfect_max_ext_angle"]: rep_feedback = "OVER EXTENDED"
                                
                                feedback_message = rep_feedback
                                feedback_display_start_time = time.time()
                                rep_history.append({"rep_num": reps_completed, "flex_angle": int(angle_at_flexion),
                                                    "ext_angle": int(angle_at_extension), "feedback": rep_feedback})
                            stage = "down"
                            angle_at_extension = current_angle
                        
                        if current_angle < params["up_angle_threshold"]: # Flexed arm
                            stage = "up"
                            angle_at_flexion = current_angle

                    elif params["movement_direction"] == "extend_to_flex": # e.g., Squats, Push-ups
                        if current_angle > params["up_angle_threshold"]: # Extended position
                            if stage == "down": # Transition from flexed to extended
                                reps_completed += 1
                                # Determine feedback
                                flex_ok = params["perfect_min_flex_angle"] <= angle_at_flexion <= params["perfect_max_flex_angle"]
                                ext_ok = params["perfect_min_ext_angle"] <= angle_at_extension <= params["perfect_max_ext_angle"]

                                rep_feedback = "GOOD!"
                                if flex_ok and ext_ok: rep_feedback = "PERFECT!"
                                elif not flex_ok and angle_at_flexion < params["perfect_min_flex_angle"]: rep_feedback = "NOT DEEP ENOUGH (FLEX)"
                                elif not flex_ok and angle_at_flexion > params["perfect_max_flex_angle"]: rep_feedback = "TOO DEEP (FLEX)" # For squats/pushups, higher angle means not deep enough
                                elif not ext_ok and angle_at_extension < params["perfect_min_ext_angle"]: rep_feedback = "NOT EXTENDED ENOUGH"
                                elif not ext_ok and angle_at_extension > params["perfect_max_ext_angle"]: rep_feedback = "OVER EXTENDED"
                                
                                feedback_message = rep_feedback
                                feedback_display_start_time = time.time()
                                rep_history.append({"rep_num": reps_completed, "flex_angle": int(angle_at_flexion),
                                                    "ext_angle": int(angle_at_extension), "feedback": rep_feedback})
                            stage = "up"
                            angle_at_extension = current_angle
                        
                        if current_angle < params["down_angle_threshold"]: # Flexed position
                            stage = "down"
                            angle_at_flexion = current_angle

                    # --- Visualize Angle ---
                    cv2.putText(image, str(int(current_angle)),
                                tuple(vertex_coords),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, WHITE, 2, cv2.LINE_AA)

                except Exception as e:
                    # print(f"Error processing landmarks: {e}")
                    pass

            # --- UI Rendering ---

            # Exercise name (top left)
            draw_text_with_background(image, current_exercise_key.replace('_', ' ').upper(), 10, 40,
                                      cv2.FONT_HERSHEY_SIMPLEX, 0.8, WHITE, GREEN, 2)

            # --- Status Box (bottom left) ---
            box_start_x, box_start_y = 0, h - 120
            box_end_x, box_end_y = 280, h # Wider box to prevent overlap
            cv2.rectangle(image, (box_start_x, box_start_y), (box_end_x, box_end_y), LIGHT_BLUE, -1)
            cv2.rectangle(image, (box_start_x, box_start_y), (box_end_x, box_end_y), BLACK, 2) # Border

            # Reps data
            cv2.putText(image, 'REPS', (20, box_start_y + 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, BLACK, 1, cv2.LINE_AA)
            cv2.putText(image, f"{reps_completed}/{target_reps}",
                        (15, box_start_y + 90),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.8, WHITE, 2, cv2.LINE_AA)

            # Stage data
            cv2.putText(image, 'STAGE', (160, box_start_y + 30), # Adjusted X position
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, BLACK, 1, cv2.LINE_AA)
            cv2.putText(image, str(stage),
                        (155, box_start_y + 90), # Adjusted X position
                        cv2.FONT_HERSHEY_SIMPLEX, 1.8, WHITE, 2, cv2.LINE_AA)
            
            # --- Progress Bar (below exercise name) ---
            progress_bar_width = w - 200 # A bit narrower
            progress_bar_height = 20
            progress_bar_x = 100
            progress_bar_y = 70
            
            cv2.rectangle(image, (progress_bar_x, progress_bar_y),
                          (progress_bar_x + progress_bar_width, progress_bar_y + progress_bar_height),
                          (100, 100, 100), 2) # Background border
            
            fill_width = int(progress_bar_width * (reps_completed / target_reps))
            cv2.rectangle(image, (progress_bar_x, progress_bar_y),
                          (progress_bar_x + fill_width, progress_bar_y + progress_bar_height),
                          BLUE, -1) # Filled progress

            # --- Dynamic Feedback Message (top right) ---
            if feedback_message and (time.time() - feedback_display_start_time < feedback_display_duration):
                text_color = GREEN if feedback_message == "PERFECT!" else YELLOW
                draw_text_with_background(image, feedback_message, w - cv2.getTextSize(feedback_message, cv2.FONT_HERSHEY_SIMPLEX, 1.0, 2)[0][0] - 20, 60,
                                          cv2.FONT_HERSHEY_SIMPLEX, 1.0, text_color, BLACK, 2)
            
            # --- Set Complete / Post-Set Summary ---
            if reps_completed >= target_reps:
                overlay = image.copy()
                alpha = 0.7 # Transparency factor

                # Darken background
                cv2.rectangle(overlay, (0, 0), (w, h), (0, 0, 0), -1)
                image = cv2.addWeighted(overlay, alpha, image, 1 - alpha, 0)

                # Set Complete Header
                draw_text_with_background(image, "SET COMPLETE!", int(w/2) - 180, int(h/2) - 150,
                                          cv2.FONT_HERSHEY_SIMPLEX, 1.8, RED, BLACK, 3)
                
                # Feedback Summary
                summary_y_start = int(h/2) - 80
                for i, rep_data in enumerate(rep_history):
                    if i >= 5: # Limit summary to first few reps or scroll later if needed
                        break 
                    summary_text = f"Rep {rep_data['rep_num']}: {rep_data['feedback']} (Flex: {rep_data['flex_angle']} Ext: {rep_data['ext_angle']})"
                    
                    text_color = GREEN if rep_data['feedback'] == "PERFECT!" else YELLOW
                    cv2.putText(image, summary_text, (int(w*0.1), summary_y_start + i * 40),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, text_color, 2, cv2.LINE_AA)

                # General Tip based on performance (simple example)
                perfect_reps = sum(1 for rep in rep_history if rep['feedback'] == "PERFECT!")
                if perfect_reps == target_reps:
                    tip_message = "Outstanding! Perfect form on all reps."
                elif perfect_reps >= target_reps * 0.7:
                    tip_message = "Great job! Minor adjustments for perfect form."
                else:
                    tip_message = "Focus on full range of motion. Review flexion/extension angles."

                cv2.putText(image, "Tip: " + tip_message, (int(w*0.1), summary_y_start + len(rep_history) * 40 + 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, WHITE, 2, cv2.LINE_AA)

                cv2.putText(image, "Press 'q' to exit.", (int(w/2) - 150, h - 50),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, WHITE, 2, cv2.LINE_AA)


            # --- Render MediaPipe Detections ---
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
                                    mp_drawing.DrawingSpec(color=(245, 117, 66), thickness=2, circle_radius=2),
                                    mp_drawing.DrawingSpec(color=(245, 66, 230), thickness=2, circle_radius=2)
                                     )

            cv2.imshow('AI Fitness Trainer', image)

            if cv2.waitKey(10) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()

C:\Users\athar\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


In [ ]:
run_camera_popup()